# Laboratorio 24: Clasificación de Cáncer de Mama
# Implementación con Regresión Logística

**Dataset:** Breast Cancer Wisconsin (Diagnostic)

**Objetivo:** Clasificar tumores como malignos o benignos basándose en características de células nucleares usando Regresión Logística.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

## 1. Carga y exploración del conjunto de datos

In [ ]:
cancer = load_breast_cancer(as_frame=True)
dfCancer = cancer.frame
dfCancer.head()

In [ ]:
print(cancer.DESCR)

In [ ]:
dfCancer.shape

In [ ]:
dfCancer.info()

In [ ]:
dfCancer.isnull().sum()

In [ ]:
dfCancer.describe()

In [ ]:
dfCancer.columns

In [ ]:
cancer.target_names

In [ ]:
dfCancer.rename(columns={"target": "diagnostico"}, inplace=True)

dfCancer["diagnostico"] = dfCancer["diagnostico"].map({
    0: "maligno",
    1: "benigno"
})

dfCancer.head()

In [ ]:
dfCancer.tail()

## 2. Variables predictoras y variable objetivo

In [ ]:
X = dfCancer.drop(columns=["diagnostico"])
y = dfCancer["diagnostico"]

print("Variables predictoras:", X.columns.tolist())
print("Variable objetivo:", y.name)

## 3. Importancia de las variables

In [ ]:
df_numeric = dfCancer.drop(columns=["diagnostico"])
df_numeric["diagnostico_num"] = (dfCancer["diagnostico"] == "benigno").astype(int)

correlaciones = df_numeric.corr()["diagnostico_num"].drop("diagnostico_num").sort_values(ascending=False)
correlaciones

In [ ]:
plt.figure(figsize=(10, 8))
correlaciones.plot(kind="barh", color="steelblue")
plt.title("Correlación de cada variable con el diagnostico (1=benigno, 0=maligno)")
plt.xlabel("Coeficiente de correlación")
plt.tight_layout()
plt.show()

## 4. Separación de datos: entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

In [ ]:
print("Distribución de clases en entrenamiento:")
print(y_train.value_counts(normalize=True))
print("\nDistribución de clases en prueba:")
print(y_test.value_counts(normalize=True))

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Implementación del modelo

In [ ]:
modelo = LogisticRegression(random_state=42, max_iter=10000)
modelo.fit(X_train_scaled, y_train)

print("Modelo entrenado correctamente")

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": modelo.coef_[0]
}).sort_values("coeficiente", key=abs, ascending=False)

print("Intercepto (beta_0):", modelo.intercept_[0])
coeficientes

## 6. Predicciones

In [ ]:
y_pred = modelo.predict(X_test_scaled)

comparacion = pd.DataFrame({
    "diagnostico real": y_test.values,
    "diagnostico predicho": y_pred
})

comparacion.head(10)

## 7. Evaluación del modelo

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy (exactitud): {accuracy:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=modelo.classes_,
    columns=modelo.classes_
)
cm_df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["maligno", "benigno"])
disp.plot(ax=ax, cmap="Blues")
plt.title("Matriz de Confusión - Regresión Logística")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
y_pred_proba = modelo.predict_proba(X_test_scaled)

proba_df = pd.DataFrame({
    "real": y_test.values,
    "predicho": y_pred,
    "prob maligno": y_pred_proba[:, 0],
    "prob benigno": y_pred_proba[:, 1]
})
proba_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test.values,
                y_pred_proba[:, 1],
                alpha=0.3, c=["green" if p == "benigno" else "red" for p in y_pred])
axes[0].axhline(y=0.5, color="black", linestyle="--", label="Umbral = 0.5")
axes[0].set_xlabel("Clase real")
axes[0].set_ylabel("Probabilidad predicha (benigno)")
axes[0].set_title("Probabilidades predichas vs Clase real")
axes[0].legend()

axes[1].barh(coeficientes["variable"].head(10), coeficientes["coeficiente"].head(10), color="steelblue")
axes[1].set_xlabel("Coeficiente")
axes[1].set_title("Top 10 coeficientes (mayor impacto)")

plt.tight_layout()
plt.show()

## 8. Implementación con datos nuevos

In [ ]:
datos_nuevos = pd.DataFrame({
    "mean radius": [14.5],
    "mean texture": [18.0],
    "mean perimeter": [93.0],
    "mean area": [650.0],
    "mean smoothness": [0.095],
    "mean compactness": [0.08],
    "mean concavity": [0.06],
    "mean concave points": [0.04],
    "mean symmetry": [0.18],
    "mean fractal dimension": [0.06],
    "radius error": [0.4],
    "texture error": [1.0],
    "perimeter error": [2.8],
    "area error": [40.0],
    "smoothness error": [0.007],
    "compactness error": [0.025],
    "concavity error": [0.03],
    "concave points error": [0.01],
    "symmetry error": [0.02],
    "fractal dimension error": [0.004],
    "worst radius": [16.0],
    "worst texture": [24.0],
    "worst perimeter": [105.0],
    "worst area": [850.0],
    "worst smoothness": [0.12],
    "worst compactness": [0.15],
    "worst concavity": [0.12],
    "worst concave points": [0.06],
    "worst symmetry": [0.25],
    "worst fractal dimension": [0.07]
})

print("Datos nuevos de entrada:")
print(f"Forma: {datos_nuevos.shape}")

In [ ]:
datos_nuevos_scaled = scaler.transform(datos_nuevos)

prediccion_nueva = modelo.predict(datos_nuevos_scaled)
probabilidad_nueva = modelo.predict_proba(datos_nuevos_scaled)

print("Predicción Regresión Logística:")
print(f"Resultado: {prediccion_nueva[0]}")
print(f"Probabilidad - maligno: {probabilidad_nueva[0][0]:.2%}, benigno: {probabilidad_nueva[0][1]:.2%}")

## Conclusiones

### Desempeño del modelo

La Regresión Logística demostró un alto rendimiento en la clasificación de tumores de mama, logrando una excelente discriminación entre casos malignos y benignos.

### Utilidad del modelo

- **Herramienta de apoyo:** Puede asistir a médicos en el diagnóstico inicial de tumores de mama.
- **Priorización de casos:** Permite identificar casos de alto riesgo que requieren atención urgente.
- **Interpretabilidad:** Los coeficientes del modelo indican qué características celulares son más relevantes para la clasificación.

### Ventajas de la Regresión Logística

- **Probabilidades:** Además de la clasificación, proporciona la probabilidad de pertenecer a cada clase.
- **Escalabilidad:** Funciona eficientemente con datasets de tamaño moderado.
- **Escalado de variables:** Al ser un modelo lineal, requiere escalado de características para un mejor rendimiento.